# Fase 3: Diseño algorítmico, Complejidad y Programación Orientada a Objetos
**Proyecto:** Análisis de datos de amenazas de ciberseguridad  
**Integrantes:** Jennifer Nilo – Patricio Núñez

---

## 1. Exploración e Ingesta Defensiva
Iniciamos configurando el entorno e importando los datos. Si el archivo masivo no está disponible (por exclusión de GitHub), el sistema cargará automáticamente nuestra muestra cruda para garantizar la reproducibilidad. Por ello, en esta fase integraremos el preprocesamiento de la Fase 2 refactorizado hacia un paradigma de Programación Orientada a Objetos (POO).

In [ ]:
import pandas as pd
import numpy as np
import timeit
import sys
import os

# Fijamos la semilla de aleatoriedad para garantizar reproducibilidad global
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Aumentamos el límite de recursión por seguridad en nuestro algoritmo Merge Sort
sys.setrecursionlimit(5000)

# Vinculamos nuestra arquitectura modular local (/src/) al path de Python
ruta_src = os.path.abspath('./src')
if ruta_src not in sys.path:
    sys.path.append(ruta_src)

def cargar_datos_crudos():
    """Carga el CSV crudo. Si no existe el masivo, usa la muestra reproducible."""
    ruta_raw = '../data/raw/cybersecurity_threat_detection_logs.csv'
    ruta_muestra = '../data/raw/cybersecurity_threat_detection_logs_muestra_500000.csv'

    if os.path.exists(ruta_raw):
        print(f"Cargando dataset masivo local...")
        df = pd.read_csv(ruta_raw)
    elif os.path.exists(ruta_muestra):
        print(f"[MODO EVALUACIÓN] Dataset masivo no detectado. Cargando muestra reproducible...")
        df = pd.read_csv(ruta_muestra)
    else:
        raise FileNotFoundError("No se encontró ningún dataset en data/raw/.")
    
    print(f"Datos cargados: {df.shape[0]:,} filas, {df.shape[1]} columnas")
    return df

# 1) Cargamos los datos crudos iniciales
df_crudo = cargar_datos_crudos()

## 2. Herencia, Polimorfismo y Encapsulamiento (POO)
En lugar de tener funciones sueltas, hemos agrupado los datos y las acciones en **Clases** dentro del archivo `src/preprocesamiento.py`. 

Aplicamos los pilares de la POO mediante el patrón *Strategy*:
* **Herencia:** Creamos una clase base `Transformador` de la cual heredan especializaciones como `LimpiadorNulos` o `EscaladorMinMax`.
* **Polimorfismo:** El orquestador no necesita saber qué hace cada transformación, solo llama al método `.aplicar()` en todas ellas por igual.
* **Encapsulamiento:** La clase `PipelinePreprocesamiento` protege el *DataFrame* operando sobre una copia interna.

In [ ]:
# Importamos nuestras clases de F3/src/preprocesamiento.py
from preprocesamiento import (
    PipelinePreprocesamiento, 
    LimpiadorNulos, 
    FiltroIPs, 
    TransformadorFechas, 
    EscaladorMinMax, 
    IngenieriaCaracteristicas
)

# 2) Armamos el pipeline
pipeline_pre = PipelinePreprocesamiento([
    LimpiadorNulos(),
    FiltroIPs(),
    TransformadorFechas(),
    EscaladorMinMax(),
    IngenieriaCaracteristicas()
])

# 3) Ejecutamos el pipeline completo en una sola línea
print("Ejecutando transformaciones orientadas a objetos...")
df_procesado = pipeline_pre.ejecutar(df_crudo)

# 4) Validamos que no queden nulos ni texto
nulos_totales = df_procesado.isnull().sum().sum()
texto_restante = df_procesado.select_dtypes(include=['object', 'string']).columns.tolist()

assert nulos_totales == 0, "Error: Quedan valores nulos."
assert not texto_restante, "Error: Quedan columnas sin codificar."

print(f"Dimensiones finales: {df_procesado.shape}")
display(df_procesado.head(3))